# 01. Specialized Language Model for Low-Resource African Language

**Course**: ICS554 Natural Language Processing · Ashesi University  
**Project**: Prosit 1 (Ankora AI Research Lab)  
**Objective**: Develop and evaluate an n-gram statistical language model for a low-resource African language, addressing data scarcity, out-of-vocabulary (OOV) words, and smoothing techniques.

---
### Pipeline Overview
1. **Data Loading & Preprocessing**: Tokenization preserving African language orthography, sentence boundary markers (`<s>`, `</s>`), and OOV handling (`<unk>`).
2. **N-Gram Model Training**: Unigram, Bigram, and Trigram count estimation.
3. **Smoothing & Backoff**: Maximum Likelihood Estimation (MLE) vs. Laplace (Add-1) vs. Linear Interpolation vs. Kneser-Ney.
4. **Intrinsic Evaluation**: Perplexity calculation on test splits.
5. **Generation**: Text sampling with temperature.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure repo root is on python path
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.preprocessing import basic_tokenize, build_vocabulary, replace_oov_tokens
from src.ngram import NGramLM
from src.viz import plot_ngram_frequency, plot_perplexity_comparison

RANDOM_SEED = 42

## 1. Data Preparation & Tokenization
We load the corpus for the chosen low-resource language (e.g. Akan/Twi, Ewe, or Yoruba).
Below is a demonstration corpus that can be swapped with raw files in `data/raw/low_resource/`.

In [ ]:
# Example sample sentences in Akan/Twi (or load from data/raw/low_resource/)
sample_corpus = [
    "Me ma wo akye",
    "Wo ho te sen?",
    "Me ho ye, medaase",
    "Kofi reko sukuu",
    "Ama to aduane pa",
    "Yenko fie ntem",
    "Osuo reto wo Nkran",
    "Okyena meba wo nkyen",
    "Abofra yi nim nyansa papaapa",
    "Yebedi nkunim wo adesua mu",
    "Osuani no sua ade yiye",
    "Kofi ne Ama koo dwam",
    "Yen nyinaa pe asomdwoe",
    "Wo din de sen?",
    "Me din de Kwabena"
]

# Tokenize
tokenized_data = [basic_tokenize(text) for text in sample_corpus]

# Train/Test split (80/20)
split_idx = int(0.8 * len(tokenized_data))
train_tokens = tokenized_data[:split_idx]
test_tokens = tokenized_data[split_idx:]

# Build vocabulary from training set only to prevent leakage
vocab, freqs = build_vocabulary(train_tokens, min_freq=1)
train_clean = replace_oov_tokens(train_tokens, vocab)
test_clean = replace_oov_tokens(test_tokens, vocab)

print(f"Vocabulary size: {len(vocab)}")
print(f"Training sentences: {len(train_clean)}, Test sentences: {len(test_clean)}")

## 2. Fitting N-Gram Models with Different Smoothing Techniques

In [ ]:
# 1. Unigram
unigram = NGramLM(n=1, smoothing="laplace").fit(train_clean, vocab=vocab)

# 2. Bigram (Laplace)
bigram_laplace = NGramLM(n=2, smoothing="laplace", k=1.0).fit(train_clean, vocab=vocab)

# 3. Bigram (Lidstone / Add-0.1)
bigram_lidstone = NGramLM(n=2, smoothing="laplace", k=0.1).fit(train_clean, vocab=vocab)

# 4. Trigram (Linear Interpolation)
trigram_interp = NGramLM(n=3, smoothing="interpolation").fit(train_clean, vocab=vocab)
trigram_interp.set_interpolation_weights([0.1, 0.3, 0.6])

# 5. Bigram (Kneser-Ney)
bigram_kn = NGramLM(n=2, smoothing="kneser_ney").fit(train_clean, vocab=vocab)

print("Models fitted successfully.")

## 3. Evaluation: Perplexity Comparison
Perplexity measures how surprised the model is by unseen test data. Lower perplexity indicates better predictive power.

In [ ]:
models = {
    "Unigram (Laplace)": unigram,
    "Bigram (Laplace)": bigram_laplace,
    "Bigram (Add-0.1)": bigram_lidstone,
    "Trigram (Interpolation)": trigram_interp,
    "Bigram (Kneser-Ney)": bigram_kn,
}

results = {}
for name, model in models.items():
    ppl = model.perplexity(test_clean)
    results[name] = ppl
    print(f"{name:25s} -> Perplexity: {ppl:.2f}")

# Plot comparison
plot_perplexity_comparison(
    list(results.keys()),
    list(results.values()),
    title="Low-Resource African Language LM - Perplexity Benchmark",
    save_path=REPO_ROOT / "figures" / "ngram_perplexity_comparison.png",
)

## 4. Text Generation
Sample generated text from the trained models.

In [ ]:
print("--- Generated Sentences ---")
for name, model in models.items():
    gen_text = model.generate(max_length=12, temperature=0.8)
    print(f"[{name}]: {gen_text}")